<a href="https://colab.research.google.com/github/2024akcs4377f-spec/ML-assignment-2/blob/cross-validation/cross_validatin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

x_train = pd.read_csv('/content/x_train.csv')
y_train = pd.read_csv('/content/y_train.csv')

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

print("\nx_train head:")
display(x_train.head())

print("\ny_train head:")
display(y_train.head())

x_train shape: (537, 8)
y_train shape: (537, 1)

x_train head:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,1,95.0,60.0,18.0,58.0,23.9,0.260,22
1,5,105.0,72.0,29.0,325.0,36.9,0.159,28
2,0,135.0,68.0,42.0,250.0,42.3,0.365,24
3,4,131.0,68.0,21.0,166.0,33.1,0.160,28
4,1,103.0,30.0,38.0,83.0,43.3,0.183,33



y_train head:


,Outcome
0,0
1,0
2,1
3,0
4,0


In [2]:
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Convert y_train to a 1D array
y_train_flat = y_train.values.ravel()

# Define the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

# Define parameter grid to search
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 5, 10, 15],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

# Setup Stratified 5-Fold Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Setup Grid Search
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Fit the model
grid_search.fit(x_train, y_train_flat)

# Print the results
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")
print("Best Parameters:", grid_search.best_params_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best CV Accuracy: 0.7821
Best Parameters: {'rf__max_depth': 10, 'rf__min_samples_leaf': 4, 'rf__min_samples_split': 2, 'rf__n_estimators': 50}


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Import the 6 models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Setup metrics to evaluate
scoring = ['accuracy', 'precision', 'recall', 'f1']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

# Run cross-validation for each model
for name, model in models.items():
    # Always scale features as part of a pipeline to prevent data leakage
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])

    cv_results = cross_validate(pipeline, x_train, y_train_flat, cv=cv, scoring=scoring, n_jobs=-1)

    results.append({
        'Model': name,
        'CV Accuracy': np.mean(cv_results['test_accuracy']),
        'CV Precision': np.mean(cv_results['test_precision']),
        'CV Recall': np.mean(cv_results['test_recall']),
        'CV F1-Score': np.mean(cv_results['test_f1'])
    })

# Create comparison table
comparison_df = pd.DataFrame(results).sort_values(by='CV Accuracy', ascending=False)
display(comparison_df)

,Model,CV Accuracy,CV Precision,CV Recall,CV F1-Score
5,Gradient Boosting,0.795171,0.716471,0.685775,0.699374
3,Random Forest,0.778366,0.704696,0.638122,0.668779
0,Logistic Regression,0.767151,0.716098,0.579516,0.637252
4,SVM,0.761648,0.702595,0.558321,0.620592
1,KNN,0.757875,0.684291,0.568137,0.617549
2,Decision Tree,0.694514,0.568938,0.563585,0.565036


In [4]:
from sklearn.model_selection import GridSearchCV

# --- 1. Tune Gradient Boosting Classifier ---
print("--- Tuning Gradient Boosting ---")
gb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('gb', GradientBoostingClassifier(random_state=42))
])

gb_param_grid = {
    'gb__n_estimators': [50, 100, 150],
    'gb__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'gb__max_depth': [3, 4, 5],
    'gb__subsample': [0.8, 1.0]
}

gb_grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
gb_grid.fit(x_train, y_train_flat)
print(f"Best Gradient Boosting CV Accuracy: {gb_grid.best_score_:.4f}")
print("Best Gradient Boosting Parameters:", gb_grid.best_params_)

# --- 2. Tune Random Forest Classifier ---
print("\n--- Tuning Random Forest ---")
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

rf_param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [5, 10, 15, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(x_train, y_train_flat)
print(f"Best Random Forest CV Accuracy: {rf_grid.best_score_:.4f}")
print("Best Random Forest Parameters:", rf_grid.best_params_)

--- Tuning Gradient Boosting ---
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best Gradient Boosting CV Accuracy: 0.7952
Best Gradient Boosting Parameters: {'gb__learning_rate': 0.1, 'gb__max_depth': 3, 'gb__n_estimators': 100, 'gb__subsample': 1.0}

--- Tuning Random Forest ---
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Random Forest CV Accuracy: 0.7821
Best Random Forest Parameters: {'rf__max_depth': 10, 'rf__min_samples_leaf': 4, 'rf__min_samples_split': 2, 'rf__n_estimators': 50}
